<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/lab-02-tokenizer-and-sampler-lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2 (graded) — Tokenizer and sampler lab
**Course 2: Generative AI and LLMs with Python — Chapter 2: Tokenization, LM objective, decoding**

**What you'll submit:** a BPE tokenizer trained on AG News, a token-count comparison across
domains, top-k and nucleus sampling implemented from scratch, a perplexity measurement, and
a decoding-strategy comparison with one example of each failure mode.

In [ ]:
!pip install -q tokenizers datasets transformers

## 1. Load AG News (with offline fallback)

In [ ]:
import numpy as np
np.random.seed(0)

def load_ag_news_texts(n=8000):
    try:
        from datasets import load_dataset
        ds = load_dataset('ag_news')
        texts = ds['train']['text'][:n]
        print(f'Loaded {len(texts)} real AG News articles.')
        return texts
    except Exception as e:
        print(f'Offline fallback engaged ({e}) — a small synthetic news-like corpus.')
        templates = [
            'Diplomats meet to discuss the ongoing crisis in the region',
            'The home team won the championship after a dramatic final',
            'Stock prices rose sharply after the quarterly earnings report',
            'Researchers unveiled a new processor with record efficiency',
        ]
        return [np.random.choice(templates) + f' (report {i})' for i in range(500)]

texts = load_ag_news_texts()
with open('ag_news_corpus.txt', 'w') as f:
    f.write('\n'.join(texts))

## 2. Train a BPE tokenizer

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

tokenizer = Tokenizer(models.BPE(unk_token='<unk>'))
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
trainer = trainers.BpeTrainer(vocab_size=4000, special_tokens=['<pad>', '<unk>', '<bos>', '<eos>'])
tokenizer.train(['ag_news_corpus.txt'], trainer)

sample = texts[0]
encoded = tokenizer.encode(sample)
print('Original:', sample)
print('Tokens:  ', encoded.tokens)
print(f'{len(sample.split())} words -> {len(encoded.tokens)} BPE tokens')

## 3. Token counts across domains — the cost implication

In [ ]:
domain_samples = {
    'news (in-domain)': texts[:20],
    'code-like': ['def forward(self, x):\n    return self.linear(x) + self.bias'] * 20,
    'repeated numbers': ['1234567890 ' * 5] * 20,
}

for domain, samples in domain_samples.items():
    avg_tokens = np.mean([len(tokenizer.encode(s).tokens) for s in samples])
    avg_words = np.mean([len(s.split()) for s in samples])
    print(f'{domain:20s}: {avg_words:5.1f} words -> {avg_tokens:5.1f} tokens '
          f'({avg_tokens / avg_words:.2f} tokens/word)')

print('\nA tokenizer trained on news text is efficient on news, and notably less efficient')
print('(more tokens per word = more $ per request) on out-of-domain text like code or numbers.')

## 4. Decoding strategies, from scratch

In [ ]:
import torch

def top_k_sample(logits, k=10):
    # TODO: zero out everything except the top-k logits, then sample from the resulting softmax
    top_vals, top_idx = torch.topk(logits, k)
    probs = torch.softmax(top_vals, dim=-1)
    choice = torch.multinomial(probs, 1)
    return top_idx[choice]

def top_p_sample(logits, p=0.9):
    """Nucleus sampling: sample from the smallest set of tokens whose cumulative
    probability exceeds p."""
    probs = torch.softmax(logits, dim=-1)
    sorted_probs, sorted_idx = torch.sort(probs, descending=True)
    cum_probs = torch.cumsum(sorted_probs, dim=-1)
    # TODO: find the cutoff index where cum_probs first exceeds p, keep that prefix,
    # renormalize, then sample. (Hint: torch.searchsorted or a manual loop both work.)
    cutoff = int((cum_probs < p).sum().item()) + 1
    keep_probs = sorted_probs[:cutoff]
    keep_probs = keep_probs / keep_probs.sum()
    choice = torch.multinomial(keep_probs, 1)
    return sorted_idx[choice]

# quick sanity check with a toy distribution
toy_logits = torch.tensor([5.0, 4.0, 1.0, 0.5, 0.1, 0.1, 0.1, 0.1])
print('top-k=3 samples:', [int(top_k_sample(toy_logits, k=3)) for _ in range(10)])
print('top-p=0.9 samples:', [int(top_p_sample(toy_logits, p=0.9)) for _ in range(10)])
print('(both should mostly pick indices 0/1, the high-probability tokens)')

## 5. Perplexity of a small pretrained model

In [ ]:
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

device = 'cuda' if torch.cuda.is_available() else 'cpu'
hf_tok = GPT2TokenizerFast.from_pretrained('gpt2')
hf_model = GPT2LMHeadModel.from_pretrained('gpt2').to(device).eval()

def perplexity(text, model, tok):
    ids = tok(text, return_tensors='pt').input_ids.to(device)
    with torch.no_grad():
        loss = model(ids, labels=ids).loss
    return torch.exp(loss).item()

for sample in texts[:3]:
    ppl = perplexity(sample, hf_model, hf_tok)
    print(f'PPL={ppl:6.1f}  "{sample[:70]}..."')

## 6. Decoding-strategy comparison, with a failure mode for each

In [ ]:
prompt = 'The company announced today that'
ids = hf_tok(prompt, return_tensors='pt').input_ids.to(device)

strategies = {
    'greedy (repetitive)': dict(do_sample=False, num_beams=1),
    'beam search': dict(do_sample=False, num_beams=4),
    'temperature=2.0 (incoherent)': dict(do_sample=True, temperature=2.0, top_k=0, top_p=1.0),
    'top-k=5': dict(do_sample=True, top_k=5),
    'top-p=0.9': dict(do_sample=True, top_p=0.9, top_k=0),
}

torch.manual_seed(0)
for name, kwargs in strategies.items():
    out = hf_model.generate(ids, max_new_tokens=25, pad_token_id=hf_tok.eos_token_id, **kwargs)
    text = hf_tok.decode(out[0], skip_special_tokens=True)
    print(f'--- {name} ---\n{text}\n')

## 7. Write-up (fill in)
For each strategy above, name the failure mode you can see (or would see with more samples):
greedy's repetition, temperature=2.0's incoherence, top-k's rigidity if k is too small, etc.
Which would you actually ship for Fernwood's use case, and why?

_Your answer here._

---
*Beacon AI · AIBits Academy — Chapter 2: Tokenization, LM objective, decoding*